# NLP Explainer - Overview

## Set up

First, make sure to install all required dependencies:
```
uv sync --extra nlp
```
or
```
pip install '.[nlp]'

We will work with the model [bhadresh-savani/distilbert-base-uncased-emotion](https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion) from Hugging Face.

This is a sentiment analysis distilbert classifier trained on [dair-ai/emotion dataset](https://huggingface.co/datasets/dair-ai/emotion)

We start by loading it with our wrapper `HFClassifierModel`.

In [ ]:
from shapash.model.hf import HFClassifierModel

model = HFClassifierModel.from_pretrained(
    "bhadresh-savani/distilbert-base-uncased-emotion",
    label_names=["sadness", "joy", "love", "anger", "fear", "surprise"],
)

We test its predict method on some selected texts:

In [ ]:
texts = [
    "i am so happy today",
    "i feel terrified",
    "when i was a child i was completely mad of my teacher who repeatedly turned my interventions to derision"
    ]
preds = model.predict(texts)
preds

## The NlpExplainer

The main object of Shapash NLP is the `NlpExplainer`. It takes as input the model and the label names.

The default explainer backend is Shap, other options include LIME and Layer Integrated Gradients. We will explore these other possibilities in a dedicated notebook.

In [ ]:

from shapash.explainer.nlp_explainer import NlpExplainer

xpl = NlpExplainer(model, label_names=["sadness", "joy", "love", "anger", "fear", "surprise"])

In [ ]:
xpl.backend

## Fit method

We start fitting the explainer with some texts of the train corpus. We will use them to find similar examples for a selected text to explain

In [ ]:
xpl.fit(texts, y=["joy", "fear", "anger"])

Looking for similar samples in the train corpus is easy. Under the hood, embeddings are computed: by default these are the inputs of the linear classification head of the model, but you can choose other layers at model initialization with the `embedding_space` parameter.

In [ ]:
xpl.find_similar("that scares me")

In [ ]:
xpl.find_similar_threshold("that scares me", threshold=0.9)

## Explain method

This is the main method of our explainer. We compute the explanations (with the selected backend, in this case Shap) for the input texts for all the classes of the model, including the predicted and the true class (given explicitly as `y`).

In [ ]:
explanations = xpl.explain(texts, y=["joy", "fear", "anger"])

The output is an immutable dataclass `NlpExplanation`, which can be saved and loaded independently from the explainer.

You can inspect the contribution values and the base values for each token

In [ ]:
# shap base values
explanations.base_values

# raw contributions for each token of the input texts: shape (n_samples, n_tokens, n_classes)
explanations.values

# token strings associated with the contribution values
explanations.token_strings

In [ ]:
explanations.texts

There are several useful methods for explanations inspection

In [ ]:
explanations.confusion_matrix()

In [ ]:
explanations.word_importance(1)

In [ ]:
explanations.word_counts()

In [ ]:
explanations.word_occurrences("happy")

Saving and loading explanations:

In [ ]:
explanations.save(my_save_path)

from shapash.explainer.nlp_explanation import NlpExplanation
explanations = NlpExplanation.load(my_save_path)

## Plots

We included several plotting functions for the explanations

In [ ]:
# bar chart of the contributions of each token, 
# the first input is the row index of the input text, the second input is the label index
explanations.plot.tokens(0, label_idx=1)

In [ ]:
# waterfall plot of the contributions of each token
explanations.plot.waterfall(1, label_idx=4)

In [ ]:
# sentence highlighting of the contributions of each token
explanations.plot.sentence(2, label_idx=3, notebook=True)

In [ ]:
# global word importance for the entire set of explanations
# the first input is the label index
explanations.plot.word_importance(1, n_top=5)

In [ ]:
# word profile for a specific word
# includes the aggregated contributions of the word for each label
explanations.plot.word_profile("terrified")

In [ ]:
# confusion matrix of the predictions
explanations.plot.confusion()

## Embedding projection and scatterplot

We can project the embeddings of the explained texts and display them on a scatter plot

In [ ]:
# Native projection with PCA (default)
scatter_xy = xpl.compute_projection(explanations)

In [ ]:
explanations.plot.scatter(scatter_xy, use_webgl=False)

You can use your own favorite projection

In [ ]:
# Custom projection (PaCMAP, UMAP, TSNE, etc.) with optional parameters
import pacmap
reducer = pacmap.PaCMAP(n_components=2, n_neighbors=1, MN_ratio=0.5, FP_ratio=2.0)

xpl.compute_projection(explanations, reducer=reducer, init="pca")

## Counterfactuals

How does the model behave by dropping some words or substituting them with others? This is the main aim of the conterfactuals explanations: 
- `AblationFlipGenerator` drops a subset of words and returns the sentences whose predicted label changed
- `HotFlipGenerator` substitutes a subset of words and returns the sentences whose predicted label changed

In [ ]:
xpl.available_cf_generators()

In [ ]:
res = xpl.generate_counterfactuals("I am happy today")

In [ ]:
import dataclasses
import pandas as pd

pd.DataFrame([dataclasses.asdict(cf) for cf in res])

In [ ]:
xpl.generate_counterfactuals(
    "I am happy today", 
    generator="ablation_flip",
    config={"max_ablations": 3}
)

## Webapp

Finally you can run the webapp, which puts all the features together and interact with

In [ ]:
app = xpl.run_app(explanations)

In [ ]:
# stop the app
app.kill()

The app has features which depend directly only from the explanations object, thus you can run it without the explainer.
Similar examples, Counterfactuals and Data Editor need the model and the explainer loaded. In this case, if you want to intereact with these features you can pass an explainer in the `engine` parameter

In [ ]:
from shapash.webapp.nlp_app import NlpWebApp
NlpWebApp(explanations, engine=xpl).run()